# Building an AI-Powered Portfolio Advisor
This notebook walks through the entire pipeline of our AI Portfolio Advisor. In a production environment, this code is split into modular Python files (like `data_fetcher.py`, `agent.py`) so that it can be easily run by a web server like Streamlit. 

Here, we've combined it all into one place so you can see exactly how the AI agent works under the hood!


In [ ]:
import os
import numpy as np
import pandas as pd
import yfinance as yf
import cvxpy as cp
from sklearn.ensemble import RandomForestRegressor
import google.generativeai as genai


## 1. Configuration & Data Fetching
We define our stock universe and create a function to fetch data. The AI agent will call this function when a user asks about market data.


In [ ]:
STOCK_SECTORS = {
    "Technology": ["AAPL", "MSFT", "NVDA"],
    "Finance": ["JPM", "BAC", "GS"],
    "Energy": ["XOM", "CVX", "COP"],
}
ALL_TICKERS = [t for tickers in STOCK_SECTORS.values() for t in tickers]

def fetch_stock_data(tickers=ALL_TICKERS, start="2019-04-01", end="2025-03-31"):
    data = yf.download(tickers, start=start, end=end, auto_adjust=False, progress=False)
    close_data = data["Adj Close"]
    if hasattr(close_data.columns, "get_level_values"):
        close_data.columns = close_data.columns.get_level_values(0)
    return close_data

# Example:
prices = fetch_stock_data()
prices.tail(3)


## 2. Risk Analysis & Optimization
Next, we need functions to calculate returns and run the Markowitz mean-variance optimization. The AI agent will call these functions when the user asks to "build a portfolio".


In [ ]:
def compute_monthly_returns(daily_prices):
    monthly = daily_prices.resample("ME").last()
    return monthly.pct_change().dropna()

def optimize_portfolio(returns, risk_tolerance="medium"):
    # Map risk tolerance to lambda (risk aversion)
    lambda_val = {"low": 6, "medium": 3, "high": 1}.get(risk_tolerance, 3)
    
    mean_returns = returns.mean().values
    cov_matrix = returns.cov().values
    cov_matrix = (cov_matrix + cov_matrix.T) / 2 # Ensure symmetry
    
    w = cp.Variable(len(returns.columns))
    
    # Constraints: sum to 1, no shorting, max 15% per stock
    constraints = [cp.sum(w) == 1, w >= 0, w <= 0.15]
    
    # Objective: maximize return - penalty * risk
    objective = cp.Maximize(mean_returns @ w - 0.5 * lambda_val * cp.quad_form(w, cp.psd_wrap(cov_matrix)))
    
    problem = cp.Problem(objective, constraints)
    problem.solve()
    
    weights = np.maximum(w.value, 0)
    weights /= weights.sum()
    
    return pd.DataFrame({"Ticker": returns.columns, "Weight": weights}).sort_values("Weight", ascending=False)

# Example:
returns = compute_monthly_returns(prices)
optimize_portfolio(returns, risk_tolerance="low").head()


## 3. The AI Agent (Tool Calling)
This is the magic part! We define a **Tool Schema** (a JSON description of our python functions) and give it to the Gemini LLM. When the user asks a question, the LLM reads the schema, realizes it needs to run a python function to get the answer, and tells us which function to run!


In [ ]:
# 1. Define the tools so the LLM understands what they do
tools = [
    {
        "function_declarations": [
            {
                "name": "fetch_market_data",
                "description": "Fetch historical stock prices.",
                "parameters": {"type": "object", "properties": {}, "required": []}
            },
            {
                "name": "optimize_portfolio",
                "description": "Run Markowitz mean-variance optimization to get portfolio weights.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "risk_tolerance": {"type": "string", "enum": ["low", "medium", "high"]}
                    },
                    "required": ["risk_tolerance"]
                }
            }
        ]
    }
]

# 2. Initialize the LLM
# NOTE: You must set your API key in the environment or pass it directly
# genai.configure(api_key="YOUR_API_KEY")

try:
    model = genai.GenerativeModel(model_name="gemini-3.6-flash", tools=tools)
    chat = model.start_chat()
    print("Agent ready!")
except Exception as e:
    print("Agent failed to load (did you set the API key?):", e)


## 4. Chatting with the Agent
Now we send a message to the agent. Notice how the agent doesn't return text immediately—instead, it returns a `function_call` request!


In [ ]:
# If you have an API key configured, uncomment this to test:

# response = chat.send_message("Can you build me a low risk portfolio?")
# print(response.candidates[0].content.parts[0].function_call)

# Notice how the LLM automatically extracts `risk_tolerance="low"` from our natural language query!
# We would then run our `optimize_portfolio` function in Python, and send the result back to the LLM to get the final answer.
